# 20.2 不重新訓練，怎麼先開啟成品？


試用模型像試騎一輛已組好的腳踏車。你不必先造輪胎，但要拿到配套零件，知道在哪裡啟動它。這一節先讓成品跑起來，後面再拆開看它怎麼學。

本版使用Qwen3-VL-2B-Instruct圖文底座與Whisper-large-v3-turbo，沒有啟用本課的微調修正。下面走已實際核對的Linux CPU路線：Bash終端機、Git、Python 3.12，以及可建立`venv`的Python。先在終端機執行`git --version`；若找不到命令，依[Git官方安裝指引](https://git-scm.com/downloads)安裝後再繼續。完整模型與配套需要數GB磁碟，運算另外需要記憶體；實測範圍見[學生操作指引](https://birdhackor.github.io/tiny-perceptron-vlm/natural-v4-student.html)第2節。[20.3](https://birdhackor.github.io/tiny-perceptron-vlm/20.3.html)會再說明權重大小與記憶體需求的差別。

先把本版程式放在新的`tiny-perceptron-natural`資料夾，保留原有專案與練習：

```bash
GIT_LFS_SKIP_SMUDGE=1 git clone --no-checkout https://github.com/birdhackor/tiny-perceptron-vlm.git tiny-perceptron-natural
cd tiny-perceptron-natural
GIT_LFS_SKIP_SMUDGE=1 git checkout --detach 59a1eda4ed7b6e8609892ec2b9013c821ac93e69
python3.12 --version
```

`GIT_LFS_SKIP_SMUDGE=1`讓Git先取得教材與程式，跳過大型訓練資料。第三行固定這次操作核對使用的程式與公開清單；現在只使用成品，還不需要下載全部練習題。最後應顯示`Python 3.12.x`。以下指令都在這個新專案的根目錄執行。

建立獨立環境，保留前面小實驗使用的`.venv`：

```bash
python3.12 -m venv .venv-natural
.venv-natural/bin/python -m pip install --upgrade pip
.venv-natural/bin/python -m pip install torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cpu
.venv-natural/bin/python -m pip install -r requirements-natural.txt
```

每個指令都寫出`.venv-natural/bin/python`，讓你不用猜正在使用哪一套Python。第三行安裝配對的CPU版PyTorch與torchvision，第四行補齊其餘固定套件。Notebook裡的這些Bash區塊是操作說明，不會自動執行下載。

接著讀取本章公開清單，再取得選定配置需要的配套：

```bash
.venv-natural/bin/python scripts/fetch_natural_release.py --manifest docs/natural-assistant/v4/public-release.json --list
.venv-natural/bin/python scripts/fetch_natural_release.py --manifest docs/natural-assistant/v4/public-release.json --output checkpoints/natural-assistant/release-v4
```

這份[公開清單](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/natural-assistant/v4/public-release.json)固定兩個官方模型的版本。本版下載步驟先取得配置說明與來源記錄，不包含另加的修正權重，也不包含完整模型權重；權重會在啟動及第一次語音辨識時另從指定官方版本取得。配置文件逐檔核對大小與內容指紋，不需要Hugging Face登入。成功後先重新核對，再啟動：

```bash
.venv-natural/bin/python scripts/fetch_natural_release.py --manifest docs/natural-assistant/v4/public-release.json --output checkpoints/natural-assistant/release-v4 --verify
.venv-natural/bin/python scripts/fetch_natural_release.py --manifest docs/natural-assistant/v4/public-release.json --output checkpoints/natural-assistant/release-v4 --serve --device cpu --dtype float32
```

第一次啟動會下載並載入固定圖文底座；語音模型等第一次按「辨識語音」才下載並載入。已有相同快取時可重用。等終端機顯示助理已啟動，再用同一臺電腦的瀏覽器開啟`http://127.0.0.1:8766/`。這個網址使用你的電腦運算；GitHub Pages提供的是教材。終端機須保持開啟，網頁才能繼續取得回答。

依序試四步，每次等結果出現再加下一種輸入：

1. 在「要送出的文字」輸入「你好，請用一句話回答」，按「送出問題」。
2. 在「照片」加入清楚的圖片，再問「照片裡有哪些東西？」接著可直接追問位置，照片會留在同一段對話。
3. 在「中文語音」選一段錄音，按「辨識語音」。這一步只產生逐字稿，尚未請聊天模型回答。
4. 讀過「辨識原稿」與「要送出的文字」。需要時更正，再按「送出問題」，確認助理回答的是你最後送出的問題。

想換一段話題，按「開始新對話」；結束時回終端機按`Ctrl+C`。這段對話上傳的圖片與聲音會清除，已下載權重仍可重用。上述文字、照片、語音聽寫、更正後送出、追問與清除對話，已用實際Chromium瀏覽器和CPU核對通路；它證明操作接通，回答品質仍看[20.13的能力卡](https://birdhackor.github.io/tiny-perceptron-vlm/20.13.html)。支援格式、容量、NVIDIA GPU安裝與離線使用，見[學生操作指引](https://birdhackor.github.io/tiny-perceptron-vlm/natural-v4-student.html)。下一節會解釋：為什麼調整小包很小，整套模型卻仍需要不少空間。
